# 3. Webアプリ作成 — 統合講義ノートブック

## この課題の目的

**「Python（Streamlit）だけで、実際に動く検索エンジンWebアプリをゼロから作れるようになる」** ことがゴールです。
座学でWebアプリとStreamlitの基礎を固めたうえで、`PROJECT ZERO`（架空企業の社内ナレッジ検索エンジン開発ミッション）を通じて、検索アプリ **Tech0 Search** を **6週間かけて1つのアプリを育てながら** 作り上げます。

---

## 全体の構成（C7：5週間で育て、6週目に発表）

| 章 | タイトル | 内容 |
|----|---------|------|
| 3.1 | この講座の前提 | Webアプリとは／フレームワーク選定 → **別冊『事前学習』** |
| 3.2 | 基本実装 | Streamlit入門・基本コンポーネント → **別冊『事前学習』** |
| **3.3** | 応用実装 — PROJECT ZERO | 検索エンジン **Tech0 Search** を1本のアプリとして育てる |
| &nbsp;&nbsp;3.3.1 | **W1** ｜ v0.1 | Googleの仕組み／Streamlit検索UI／**簡易検索（部分一致）** ＋「本文まで検索したいが本文が無い→どう集める？」の課題提起 |
| &nbsp;&nbsp;3.3.2 | **W2** ｜ v0.2 | Webクローラー（自動登録）で本文まで **pages_w2.json に自動収集**（検索はW1の部分一致を流用） |
| &nbsp;&nbsp;3.3.3 | **W3** ｜ DB化＋全文検索 | JSON → SQLite へ移行し、**本文まで全文検索**（DB化のあとに全文検索） |
| &nbsp;&nbsp;3.3.4 | **W4** ｜ v1.0 | TF-IDF ランキング（宿題ここまで） |
| &nbsp;&nbsp;3.3.5 | **W5** | 役員会に向けた発表準備 |
| &nbsp;&nbsp;3.3.6 | **W6** | 最終発表 |

> 📘 **事前学習（別冊）**：Webアプリの前提と Streamlit の基礎は `事前学習_Webアプリとフレームワーク・Streamlit基礎.md` にまとめています。本編に入る前に、まずそちらで基礎を固めてください。

3.1・3.2で学んだStreamlitとPythonの基礎が、そのまま実戦で活きてきます。


---
# 3.3 応用実装 — PROJECT ZERO

ここからは応用実装として、実践プロジェクト **PROJECT ZERO** に取り組みます。
架空の東証プライム上場企業「株式会社テクゼロンBANK」のCDO（最高デジタル責任者）**黒崎** からの指令を受け、社内ナレッジ検索エンジン **Tech0 Search** を、**1つのアプリとして6週間かけて段階的に育て**ていきます。

```
3.3.1 W1 ｜ v0.1  Streamlit検索UI × 簡易検索（部分一致）
        ↓  「タイトルしか検索できないな。本文まで探せ。だが本文はどう集める？」
3.3.2 W2 ｜ v0.2  Webクローラー（自動で pages_w2.json に集める）
        ↓  「JSONに貯め続けるのは限界だ。ちゃんとした器（DB）に移せ」
3.3.3 W3 ｜ DB化＋全文検索  JSON→SQLite に移し、本文まで全文検索する
        ↓  「並び順が甘い。Googleのように賢く並べろ」
3.3.4 W4 ｜ v1.0  TF-IDF ランキング（宿題はここまで）
        ↓  「役員会で説明できる状態にしろ」
3.3.5 W5        発表準備（ROI・Q&A・デモ）
3.3.6 W6        最終発表
```

> 💡 **育てる開発**：毎週ゼロから作り直すのではなく、**前週のアプリ（app.py）に機能を足していく**のがこのプロジェクトの肝です。W1で作った画面は最後まで生き続けます。


---
## 3.3.1 W1 ｜ Tech0 Search v0.1


# 導入：Googleの仕組みを知る（W1の入口）

検索エンジンを作る前に、**世界最大の検索エンジン=Google が何をしているか** をざっくり掴みましょう。
細部は不要です。「検索エンジンは3つの仕事をしている」という骨格だけ持ち帰ってください。

| 仕事 | Googleでは | このプロジェクトでは |
|------|-----------|---------------------|
| ① **集める**（クロール） | 世界中のWebページをBotが巡回して集める | W2 で **クローラー** を作る |
| ② **貯める・整理する**（インデックス） | 集めたページを検索しやすい形で保存 | W1 は JSON、W3 で **DB（SQLite）** に移行 |
| ③ **探して並べる**（ランキング） | キーワードに合うページを賢い順で返す | W1 は部分一致 → W3 で **全文検索** → W4 で **TF-IDF** |

```
① 集める（クローラー）  →  ② 貯める（インデックス/DB）  →  ③ 探して並べる（検索・ランキング）
```

> Google検索の仕組み（クローラ／インデックス／ランキング）については、「①集める→②貯める→③並べる」の3段構えだけ覚えれば十分です。
>
> **W1のゴール**：まず ③ の入口＝**Streamlitで検索できる画面**を作り、「タイトルだけの部分一致」でどこまでできるか／何が足りないかを体感します。足りない部分（本文をどう集めるか＝①）が、次週 W2 のクローラーにつながります。


# PROJECT ZERO — W1

### 〈PROJECT ZEROとは〉

あなたは今、ある特命プロジェクトのメンバーです。

> 株式会社テクゼロンBANK（東証プライム上場）  
> 経常収益：1,500億円 ／ 従業員数：3,000名 ／ 店舗数：150箇所  
> 主力事業：銀行業（リテール・法人金融）  
> 課題：次の10年の柱となる新規事業

CDO（最高デジタル責任者）の **黒崎** から、あなたに指令が届いています。

---

## CDOからの指令書 — W1 指令

```
╔════════════════════════════════════════════╗
║  W1 指令書                PROJECT ZERO 　　　║
╠════════════════════════════════════════════╣
║                                            ║
║  事前の調査報告、読んだ。よく調べた。            ║
║                                            ║
║  だが、資料だけでは役員は動かない。              ║
║  「動くもの」を見せないとな。                   ║
║                                            ║
║  今週は検索画面を作れ。シンプルでいい。          ║
║                                            ║
║  Googleみたいに、キーワードを入れたら           ║
║  結果が出てくるやつだ。                        ║
║  社内のナレッジで検索できるようにしろ。           ║
║                              CDO 黒崎       ║
╚════════════════════════════════════════════╝
```

---

### 〈CDOの言葉 → 技術要件への翻訳〉

| CDOの発言 | 技術要件 |
|-----------|----------|
| 「キーワードを入れたら結果が出る」 | 検索ボックス + 部分一致検索 |
| 「（検索対象は）社内のナレッジ」 | 行内ナレッジのページ情報をJSONで管理 |
| 「シンプルでいい」 | 最低限のUI（Streamlit標準）|

---

### 〈今週の成果物〉

| 項目 | 内容 |
|------|------|
| 技術 | Python, Streamlit, JSON |
| 機能 | 検索ボックス + 結果表示 + 登録機能 |

---

### このノートブックの使い方

**セットアップ手順・フォルダ構成・よくあるエラーの対処は `README.md` を参照してください**
**`pages_w1.json` はあらかじめ配布済みです。このノートブックと 同じフォルダ に置いてください**

#### 「写経」はどこに書く？

| 作業 | 書く場所 | やり方 |
|------|----------|--------|
| **■ 写経（1）〜（7）** | **このノートブックのコードセル** | すでに書いてあるコードセルを **そのまま実行（▶）** し、出力を確認。余裕があれば下に新しいセルを作って**自分の手で打ち直す**と定着します |
| **`search.py` を作る** | 配布の **`search.py`（虫食い）** | `____` を埋める → 詰まったら **`search_answer.py`** と見比べる（Step 4 のまとめで使用） |
| **`app.py` を作る** | **`tech0-search/` フォルダを自分で作って新規作成** | Step 5 の骨格を見ながら自分で書く → 詰まったら **`answers/w1/`** の答えを見る （※`1_事前学習/` フォルダの `app.py` は**Streamlit 練習用の別ファイル**です。W1 の `app.py` はここで新しく書きます） |


> **！大事なこと！**  
> 各週の完成コードは **`answers/` フォルダ** に入っています（w1〜w4）。  
> ただし、**まず自分で書く → 動かない → 答えと見比べる** の順で使ってください。  
> **動いた瞬間の感動を、ぜひ自分のものにしてください！**


---
# Step 0：UI設計 — 今週作る「完成画面」を掴む

> ### 🎯 〈このStepのねらい〉
>
> 作るものの「完成形」を先に決めておかないと、**どのコードを書けばいいか**が決まりません。
> Step 0 はコードを1行も書かずに、今週のゴール画面を頭に入れる時間です。
>
> **このStepが終わったときにできていること**
> - 今週作る画面を分解して説明できる
> - CDOの指令（「社員が探せる仕組みを作れ」）が、画面のどの部分に対応するか言える
>
> **触るファイル**：なし（紙とペンだけ。コードは Step 2 から）

### まずはコードを書く前に「完成画面」をイメージしよう！

エンジニアはコードを書き始める前に、まず **「どんな画面を作るか」** を設計します。  
頭の中のイメージを先に整理しておくと、「何のコードを書けばいいか」が明確になります。



✏️ **まずは自分でイメージしてみよう！**  

In [ ]:

CDOの指令を読み返して、**紙や手帳に「自分の思う完成画面」を手書きで書いてみてください。**  
正解はありません。箱を書くだけでもOKです。  

書けたら次に進みます。次のトグルで「今週作る完成形（＝ゴール）」をお見せします。


### 〈完成画面 ＝ 今週作るゴール〉

<details>
<summary><b>クリックして完成画面を見る</b></summary>

**これが今週の完成形（ゴール）です。まずはこの1枚を頭に焼き付けましょう。**
**いきなり全部は作りません。この画面を「見た目 → 中身」の順で少しずつ組み立てます。**

![Tech0 Search v0.1 完成画面](pic/w1_hero.jpg)

</details>


---

### イメージは画面を3つのパーツに分解すること！

```
┌──────────────────────────────────────────────┐
│  ① タイトルエリア                              │
│     Tech0 Search v0.1                        │
│     PROJECT ZERO — 社内ナレッジ検索エンジン      │
├──────────────────────────────────────────────┤
│  ② タブエリア                                 │
│     [ 検索 ]  [ 登録 ]  [ 一覧 ]               │
├──────────────────────────────────────────────┤
│  ③ コンテンツエリア（タブごとに中身が変わる）      │
│     検索タブ → 入力ボックス + 結果カード          │
│     登録タブ → 入力フォーム                     │
│     一覧タブ → 全件リスト                       │
└──────────────────────────────────────────────┘
```

この後は、いきなりコードから作りません。  
まず **Step 1 で「全体像」を抽象でつかみ**、そのあと **ファイルごと（`app.py` → `pages_w1.json` → `search.py`）** と、実際のエンジニアに近い順番で組み立てていきます。


---

## ✅ Step 0 まとめ

**このStepでやったこと**

| やったこと | ひとことで言うと |
|------------|------------------|
| 完成画面を自分でイメージした | 正解を見る前に、まず自分の頭で考えた |
| ゴール画面を確認した | 今週の到達点が具体的な絵になった |
| 画面を3つのパーツに分解した | ① タイトル ② タブ ③ コンテンツ |

**いまの成果物の状態**

| ファイル | 状態 |
|----------|------|
| `app.py` | ⬜ 未着手 |
| `pages_w1.json` | ⬜ 未着手 |
| `search.py` | ⬜ 未着手 |

**➡️ 次は Step 1**：その画面の **裏側で何が起きているか** を、コードを見ずに言葉で追いかけます。


---
# Step 1：作るものの「全体像」を、まず“抽象”でつかむ

> ### 🎯 〈このStepのねらい〉
>
> Step 0 で「どんな画面か」は掴めました。次は **「その裏で何が起きているか」**。
> ファイル名やコードから始めると迷子になるので、先に全体像を上から眺めます。
>
> **このStepが終わったときにできていること**
> - 検索の裏側の流れを、**固有名詞を一切使わず** 説明できる
> - 3つの役割（見た目／データ／処理）と 3つのファイルの対応が言える
> - `tech0-search/` のフォルダ構成を、何も見ずに書ける
>
> **触るファイル**：なし（設計だけ。手を動かすのは Step 2 から）

### 〈このStepの進め方〉具体的なコードに入る前に、“全体像”を上から眺める

いきなりファイルやコードから始めると、「いま何を作っているのか」を見失いがちです。  
そこで **Step 1 では、コードもファイル名もいったん脇に置き**、  
「このアプリは “結局のところ何をしているのか” 」を **抽象度の高い全体像** としてつかみます。

- **1-1**　完成画面の裏で起きること ＝ **データの流れ**（固有名詞なし）
- **1-2**　その流れを担う **3つの役割**（見た目／データ／処理）
- **1-3**　役割を **実際のファイル** に紐づける（＝設計図・関心の分離）

> **このあとの進め方**：Step 1 で「全体像（抽象）」をつかんだら、  
> Step 2 以降は **ファイルごと**（`app.py` → `pages_w1.json` → `search.py`）に、  
> 「そのファイルが全体像のどこを担うのか」を確かめながら実装していきます。


### 1-1. 完成画面の「裏側」で何が起きているか（＝データの流れ）

Step 0 で完成画面を見ました。検索ボックスに言葉を入れると、結果カードが並ぶ。  
その **裏側で何が起きているべきか** を、**コードもファイル名も出さず、言葉と矢印だけ** で追ってみます。

```
① ユーザーが検索ボックスに言葉を入力する（例：「DX」）
        ↓
② その言葉を受け取る
        ↓
③ 保存してある「ページのデータ」と、1件ずつ照らし合わせる
        ↓
④ 言葉が含まれるページだけを取り出す
        ↓
⑤ 取り出した結果を画面に返す（→ 結果カードとして並ぶ）
```

※ ここではまだ **ファイル名も関数名も出しません。**「どういう順で何が起きるか」だけ掴めればOKです。  
（この固有名詞ゼロの流れが、このアプリの「要件」です。）

---

### 1-2. この流れを担う「3つの役割」に分解する

①〜⑤をよく見ると、性質のちがう **3つの役割** が必要だと分かります。

| 役割 | 全体像での担当（①〜⑤のどこか） | ざっくり言うと |
|------|--------------------------------|----------------|
| 🖼 **見た目**            | ①入力を受け取り／⑤結果を画面に返す         | ユーザーが触る画面 |
| 🗄 **データの置き場所**  | ③で照らし合わせる「ページのデータ」を貯める | 情報の保管庫 |
| 🔧 **照らし合わせる処理**| ②〜④の「探して取り出す」を実行する         | 検索エンジンの中身 |

> 💡 **ポイント**：ここまで一度も「app.py」「json」「関数」といった言葉を使っていません。  
> **まず「何が必要か（役割）」を先に決め、「それを何で作るか（実装）」は次に決める。** これが設計の順番です。


---

### 1-3. 3つの役割を「実際のファイル」に紐づける（設計図）

役割が3つに分かれました。ここで初めて **実際のファイル名** に紐づけます。  
1つのファイルに全部書くと 100〜200行になって読めなくなるので、**役割ごとにファイルを分けます**。　

これをコンビニで例えると、

| 役割 | ファイル | コンビニで例えると | このあと作る Step |
|------|----------|-------------------|-------------------|
| 🖼 見た目             | `app.py`     | 店舗の売り場レイアウト     | **Step 2** |
| 🗄 データの置き場所   | `pages_w1.json` | バックヤードの在庫棚       | **Step 3** |
| 🔧 照らし合わせる処理 | `search.py`  | レジの計算システム         | **Step 4** |

役割ごとに分けることで「どこを直せばいいか」が一目でわかります。  
→ エンジニアはこれを **「関心の分離」** と呼びます。

---

### 〈フォルダ構成（今週の完成形）〉

```
tech0-search/
├── app.py          # 見た目（Streamlit画面）    ← Step 2 で作る
├── search.py       # 検索ロジック              ← Step 4 で作る
├── pages_w1.json      # ページデータ（データの器）  ← Step 3 で作る
├── requirements.txt
└── README.md
```

> 次のセルからは **Step 2**。全体像の「🖼 見た目」にあたる **`app.py`** から作り始めます。  
> どの画面パーツがどのコードに対応するかは、次の **【画面 × コード 対応マップ】** で確認します。


---

## ✅ Step 1 まとめ

**このStepでやったこと**

| やったこと | ひとことで言うと |
|------------|------------------|
| データの流れ①〜⑤を追った | 固有名詞ゼロで書いた「要件」 |
| 3つの役割に分解した | 🖼 見た目／🗄 データ／🔧 処理 |
| 役割をファイルに紐づけた | `app.py` / `pages_w1.json` / `search.py` ＝ 関心の分離 |
| フォルダ構成を確認した | 今週の完成形の見取り図 |

**いまの成果物の状態**

| ファイル | 状態 |
|----------|------|
| `app.py` | ⬜ 未着手（次の Step 2 で着手） |
| `pages_w1.json` | ⬜ 未着手 |
| `search.py` | ⬜ 未着手 |

**➡️ 次は Step 2**：設計図の「🖼 見た目」担当、`app.py` を書き始めます。ここから手が動きます。


---
# Step 2：【ファイル別①】`app.py` — まず「見た目」だけ作る（この段階では動かなくてOK）

> ### 🎯 〈このStepのねらい〉
>
> Step 1 で分けた3つの役割のうち、**🖼 見た目 ＝ `app.py`** から作ります。
> 器を先に作ってしまえば、「あとは何を用意すれば動くのか」が自分の目で見えます。
>
> **このStepが終わったときにできていること**
> - タイトル・タブ・検索ボックスが **ブラウザに表示される**
> - `st.title()` `st.tabs()` `st.text_input()` など主要コマンドの役割が言える
> - 「検索しても本物の結果が出ない」理由を、**自分の言葉で** 説明できる
>
> **触るファイル**：`app.py`（見た目だけ。検索結果はダミーを固定表示）

### なぜ「見た目」から先に作るのか？

料理でも、いきなり火にはかけません。まず **完成した盛り付けをイメージし、器を並べる** ところから始めます。  
アプリも同じで、先に **「見た目（画面）」だけ** を組んでしまえば、「あとは何を作れば動くのか」がハッキリ見えてきます。

この Step のゴールは **「動く」ことではありません。**

- ✅ タイトル・タブ・検索ボックスなど **画面の枠（見た目）** が出ればOK  
- ⏳ 検索しても **本物の結果はまだ出ません**（結果カードは「ダミー＝仮の見本」を置いておきます）

> **この後の流れ**  
> **① 見た目を作る（今ここ） → ② 動かす → ③「動かない＝中身が無い」を体感 → ④ 中身（データ・ロジック）を作って本物に差し替える**


---
# 見た目を作る道具 ＝ Streamlit とは？

### Streamlit の基本

**Streamlit（ストリームリット）は「Python コードだけで Web アプリを作れるライブラリ」です**

| 普通の Web アプリ開発 | Streamlit なら |
|----------------------|---------------|
| HTML＋CSS＋JavaScriptの3言語が必要 | **Python だけ** で OK |
| 最低でも数週間〜数ヶ月の学習が必要 | 今日から始められる |

**Streamlit の最大の特徴は「コードを上から下に書くだけで画面を作れる」ことです**

```python
# これだけで画面が作れる！
import streamlit as st
st.title('Hello, Tech0!')   # → 大きいタイトルが画面に表示される
st.write('こんにちは')      # → テキストが表示される
```

> ⚠️ **注意**：Streamlit のコードはこのノートブック内では画面表示できません。  
> 「どんなコードを書けば何が表示されるか」を理解することが目的です。  
> 実際の動作確認は自分で `app.py` を作って `streamlit run app.py` で行います。


## 【画面 × コード 対応マップ】— どのパーツがどのコードでできているか

**画面の各パーツが、どの Streamlit コードで作られているかを一枚にまとめた「地図」です。**　　

**まずは 上半分（タイトル・タブ・検索ボックス） に注目してください。ここを最初に作ります**　　

**下半分（検索結果カード）は「中身（ロジック）」が必要なので、あとで作ります**


![画面×コード対応マップ](pic/w1_codemap.png)


> 💡 **ポイント**  
> 全部を一度に覚える必要はありません。  
> 「この部分を変えたい」と思ったとき、この図に戻って確認しましょう。


## Streamlit 主要コマンド早見表

**app.py を自分で書くときに必要なコマンドをまとめました。**　　

**ここでは「読むだけ」で大丈夫です。実際の動作は `app.py` に書いて試しましょう**

---

### 【表示系】テキストを画面に出す
```python
st.title('大きなタイトル')         # 一番大きい見出し    → 「Tech0 Search v0.1」
st.subheader('中見出し')          # 中くらいの見出し    → 「キーワード検索」
st.caption('小さい補足テキスト')   # 小さいテキスト      → 「PROJECT ZERO —...」
st.markdown('**太字** や *斜体*') # Markdown記法で装飾 → ハイライトされた説明文
st.divider()                    # 横線を引く         → 検索結果の区切り線
```

---

### 【入力受付系】ユーザーの操作を受け取る
```python
query = st.text_input('キーワードを入力')      # テキスト入力ボックス → 検索ボックス
text  = st.text_area('説明')                 # 複数行テキスト入力   → 登録フォームの説明欄
opt   = st.selectbox('カテゴリ', ['A', 'B'])  # ドロップダウン選択   → カテゴリ選択
```

---

### 【レイアウト系】画面の構造を作る
```python
# タブを作る
tab1, tab2, tab3 = st.tabs(['検索', '登録', '一覧'])
with tab1:
    st.write('検索タブの中身')   # インデント内が検索タブの内容になる

# 横2列に分割する → 作者と日付を左右に並べる
col1, col2 = st.columns(2)
col1.caption('田中太郎')
col2.caption('2025-01-15')

# アコーディオン（クリックで開閉）
with st.expander('営業店DX推進レポート'):
    st.write('中身')
```

---

### 【フォーム系】まとめて送信できる入力エリアを作る
```python
with st.form('register_form'):           # フォームを作る
    url       = st.text_input('URL')
    title     = st.text_input('タイトル')
    submitted = st.form_submit_button('登録する')   # 送信ボタン

if submitted:                             # 送信ボタンが押されたとき
    st.success('登録完了！')              # 緑の成功メッセージ
    st.error('URLは必須です')             # 赤のエラーメッセージ
```

---

### 【セッション状態】画面を作り直しても消えない入れ物
```python
# Streamlit は操作のたびに app.py を先頭から実行し直す。
# 普通の変数は毎回リセットされるが、st.session_state に入れたものだけは残る。

st.session_state['registered'] = True      # 値を入れる（辞書と同じ書き方）
st.session_state.get('registered')         # 値を読む（無ければ None）

# 使いどころ：st.rerun() をまたいでメッセージを渡す
#   保存側 → st.session_state['registered'] = True ／ st.rerun()
#   表示側 → if st.session_state.get('registered'): st.success('登録完了！')
```
※ 詳しくは **Step 5-1** で解説します。

---

### 【画面の作り直し】
```python
st.rerun()   # 実行をここで打ち切り、app.py を先頭から走り直す
             # ⚠️ これより前に表示したもの（st.success など）は消える
```

---

### 【キャッシュ】データ読み込みを高速化する
```python
@st.cache_data                    # このデコレータを付けるだけ
def load_pages():
    with open('pages_w1.json', ...) as f:
        return json.load(f)
# 1回目だけファイルを読んで、2回目以降はメモリの値を使い回す（高速化）
# 登録後にデータを更新するには st.cache_data.clear() でキャッシュをリセット
```


## ■ 写経 （0）— 見た目だけの `app.py` を書いてみよう

In [ ]:

**下のコードを `app.py` に書いて、ターミナルで `streamlit run app.py` を実行して開いてみましょう**
**ポイントは 検索結果を「ダミー（仮の見本カード）」で固定表示 していること**
**＝ まだ検索ロジックが無くても、画面の見た目は先に完成させられます**


# app.py（見た目だけ版）— ロジックはまだ書かない
import streamlit as st

# ① タイトルエリア
st.title("Tech0 Search v0.1")
st.caption("PROJECT ZERO — 社内ナレッジ検索エンジン")

# ② タブエリア
tab1, tab2, tab3 = st.tabs(["検索", "登録", "一覧"])

# ③ コンテンツエリア（検索タブ）
with tab1:
    query = st.text_input("キーワードを入力")   # 検索ボックス（見た目だけ）

    st.divider()

    # ▼▼▼ ここは今は「ダミー（仮の見本）」です ▼▼▼
    #     後ほど Step5 で、本物の search_pages(query, pages) の結果に差し替えます
    st.caption("※ これは仮の表示です（まだ本物の検索は動きません）")
    with st.expander("📄 営業店DX推進レポート"):
        st.markdown("窓口業務のペーパーレス化で事務時間を30%削減した**DX**プロジェクトの記録")
        st.caption("👤 田中太郎　📅 2025-01-15")
    # ▲▲▲ ダミーここまで ▲▲▲

with tab2:
    st.write("（登録フォームは後で作ります）")

with tab3:
    st.write("（一覧は後で作ります）")


> 💡 `query` に何を入力しても、いまは同じダミーカードが出るだけでOKです。  
> 「見た目の枠」ができていることを確認しましょう。


---
### 動かしてみよう → そして「動かない」を確かめる

ターミナルで `streamlit run app.py` を実行し、ブラウザで開いてみましょう。

| 確認すること | 期待する状態 |
|--------------|--------------|
| タイトル「Tech0 Search v0.1」が出ているか | ✅ 出る |
| タブ（検索／登録／一覧）が3つ並んでいるか | ✅ 出る |
| 検索ボックスに文字が入力できるか | ✅ 入力できる |
| 検索して **本物の結果** が出るか | ❌ 出ない（ダミーのまま） |

**見た目はできました。でも「本物の検索」はまだ動きません。**  
理由はシンプルで、**「照らし合わせるデータ」も「探す処理」もまだ無い** からです。  
→ ここからが本番。次の Step で、その「中身」を作っていきます。


⚠️〜もしエラーが出たら〜

まず、**カレントディレクトリが `app.py` のあるディレクトリになっているか** を確認してください。

```bash
pwd   # 今いる場所はどこか
ls    # app.py が見えるか
```


---

## ✅ Step 2 まとめ

**このStepでやったこと**

| やったこと | ひとことで言うと |
|------------|------------------|
| Streamlit の役割を知った | Pythonだけで画面が作れる道具 |
| 画面×コード対応マップを見た | どのパーツがどのコードでできているか |
| 写経（0）で見た目を作った | タイトル・タブ・検索ボックス |
| 「動かない」を確かめた | **中身が無いから動かない** と体感した |

**いまの成果物の状態**

| ファイル | 状態 |
|----------|------|
| `app.py` | 🟡 **見た目だけ完成**（検索結果はダミーのまま） |
| `pages_w1.json` | ⬜ 未着手（次の Step 3 で着手） |
| `search.py` | ⬜ 未着手 |

**➡️ 次は Step 3**：動かなかった原因の1つ目、**照らし合わせるデータ**（`pages_w1.json`）を用意します。


---
# Step 3：【ファイル別②】`pages_w1.json` — データの器を作る

> ### 🎯 〈このStepのねらい〉
>
> Step 2 で画面はできましたが、**照らし合わせる相手のデータがありません**。
> Step 1 の「🗄 データの置き場所」＝ `pages_w1.json` を、ここで用意します。
>
> **このStepが終わったときにできていること**
> - 1件分のページを **辞書（dict）**、複数件を **リスト（list）** で書ける
> - 辞書のキー（title / description / keywords …）が **画面のどこに出るか** 対応づけられる
> - `json.load()` で読み込み、`json.dump()` で書き込みができる
>
> **触るファイル**：`pages_w1.json`
> （このStepでは構造と読み書きを **Jupyter 上で** 理解します。実ファイルは配布物をそのまま使います）

## 3-1. JSONとは何か？

**JSON（ジェイソン）は「データを保存するためのテキストファイルのルール」です。**　

**Excelの表と同じように「データを整理して保存する入れ物」ですが、プログラムから読み書きしやすい形式です。**

**📌 今回のアプリでは `pages_w1.json` に テクゼロンBANK の行内ナレッジ（業務資料・事例ページ）の情報を保存しており、**
**登録タブから新しいページを追加すると、このファイルに追記されます。**

**まず「辞書（dict）」という概念を理解しましょう。**

**辞書とは「キー: 値」のペアでデータを整理するPythonの入れ物です。**


## ■ 写経（1）— 1件分のページデータを辞書で書く

**`pages_w1.json` の1件分のデータは、Python の「辞書（dict）」として表現されます。**

**キーと値のペアを `"キー": 値` の形で並べることで、1件分のナレッジページ情報を表します**

**このコードでやること：**
- `page = { ... }` という辞書を書いて実行する
- `page["title"]` のようにキーを指定して値を取り出す
- `page["keywords"][0]` でリストの中の最初の要素だけ取り出す

```
下記のコードセルを実行すると、こんな出力が得られます：
タイトル  ： 営業店DX推進レポート
作者      ： 田中太郎
キーワード： ['DX', '営業店', '業務効率']
最初のKW  ： DX
```

**使う知識：** 辞書（dict）の定義 / `dict["キー"]` での値の取り出し / リストのインデックス参照


In [1]:
# pages_w1.json の 1件分のデータを Python の「辞書（dict）」で書くとこうなる
# 「キー」: 「値」 の形でデータを整理する

page = {
    "id"         : 1,
    "title"      : "営業店DX推進レポート",
    "url"        : "https://kb.techzeron-bank.example/dx-report",
    "description": "窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録",
    "keywords"   : ["DX", "営業店", "業務効率"],   # ← [ ] はリスト（複数の値）
    "author"     : "田中太郎",
    "created_at" : "2025-01-15",
    "category"   : "事例",
}

# キーを指定して値を取り出す
print("タイトル  :", page["title"])
print("作者      :", page["author"])
print("キーワード:", page["keywords"])
print("最初のKW  :", page["keywords"][0])  # ← リストの0番目 = 最初の要素

タイトル  : 営業店DX推進レポート
作者      : 田中太郎
キーワード: ['DX', '営業店', '業務効率']
最初のKW  : DX


### この辞書の各キーは、画面のどこに表示されるの？

```
┌──────────────────────────────────────────────────────────┐
│                                                          │
│  ###  📄 営業店DX推進レポート   ← page["title"]             │
│                                                          │
│  窓口業務のペーパーレス化で…**DX**プロジェクトの記録            │
│                           ↑ page["description"]          │
│                             （DXが太字なのはハイライト処理）  │
│                                                          │
│  🏷 `DX`  `営業店`  `業務効率`  ← page["keywords"]          │
│                                                          │
│  👤 田中太郎         ← page["author"]                      │
│  📅 2025-01-15       ← page["created_at"]                │
│                                                          │
│  🔗 https://kb.techzeron-bank.example/dx-report ← page["url"] │
│                                                          │
└──────────────────────────────────────────────────────────┘
```

**辞書の各「キー」が、画面のパーツと1対1で対応しています！**


## ■ 写経（2）— 複数件をリストにまとめる（pages_w1.json の全体像）

**複数の辞書を「リスト（list）」にまとめると、`pages_w1.json` の全体像になります。**

**リストとは `[ ]` の中にデータを `,` で区切って並べたものです**

**このコードでやること：**
- 辞書を複数まとめたリスト `pages = [ {...}, {...}, ... ]` を書いて実行する
- `len(pages)` で件数を確認する
- `for page in pages:` でループして全件のタイトルを表示する

```
実行すると、こんな出力が得られます：
登録件数： 3
1件目： 営業店DX推進レポート
2件目： 住宅ローン審査AI導入の検討
3件目： 法人融資 稟議書の書き方ガイド
```

**使う知識：** リスト（list）の定義 / `len()` / for ループ

**⚠️ Python の添字（番号）は 1ではなく、0から始まる ことに注意！**


In [2]:
# 複数件のページをリストにまとめる = pages_w1.json の全体像

pages = [
    {   # 1件目
        "id": 1,
        "title": "営業店DX推進レポート",
        "description": "窓口業務のペーパーレス化で事務時間を30%削減",
        "keywords": ["DX", "営業店"],
        "author": "田中太郎",
    },
    {   # 2件目
        "id": 2,
        "title": "住宅ローン審査AI導入の検討",
        "description": "AI事前審査で融資業務をDX化する企画提案",
        "keywords": ["AI", "融資", "審査"],
        "author": "鈴木花子",
    },
    {   # 3件目
        "id": 3,
        "title": "法人融資 稟議書の書き方ガイド",
        "description": "法人融資の稟議書作成の標準手順とチェックリスト",
        "keywords": ["融資", "稟議", "法人"],
        "author": "山本一郎",
    },
]

print(f"登録件数      : {len(pages)} 件")          # len() でリストの長さを取得
print(f"1件目タイトル : {pages[0]['title']}")      # 添字 0 = 最初
print(f"2件目タイトル : {pages[1]['title']}")      # 添字 1 = 2番目
print(f"3件目タイトル : {pages[2]['title']}")      # 添字 2 = 3番目（0始まりに注意！）
print()
# 一覧表示
print("── 全件タイトル一覧 ──")
for i, p in enumerate(pages):           # enumerate で「番号付き」で取り出す
    print(f"  {i+1}件目: {p['title']}")

登録件数      : 3 件
1件目タイトル : 営業店DX推進レポート
2件目タイトル : 住宅ローン審査AI導入の検討
3件目タイトル : 法人融資 稟議書の書き方ガイド

── 全件タイトル一覧 ──
  1件目: 営業店DX推進レポート
  2件目: 住宅ローン審査AI導入の検討
  3件目: 法人融資 稟議書の書き方ガイド


## 3-2. JSONファイルを「読み込む」「書き込む」

**プログラムを終了するとメモリ上のデータは消えてしまいます。**

**ファイルに保存することで「アプリを再起動してもデータが残る」ようになります。**

**→ これを データの永続化（えいぞくか） といいます。**

## ■ 写経（3）— JSONファイルを読み込む

**このコードでやること：**
- `json.load()` を使って `pages_w1.json` をファイルから読み込む
- 読み込んだデータが「辞書のリスト」として扱えることを確認する
- 件数と最初の1件のタイトルを表示する

```
実行すると、こんな出力が得られます：
読み込み件数： 5件
```

**使う知識：** `import json` / `with open(...) as f:` / `json.load(f)`


In [3]:
# json ライブラリ：Python に最初から入っているJSONを扱うための道具
# pip install 不要！
import json

# ── JSONファイルを「読み込む」──
# これが app.py の load_pages() 関数がやっていることです

# ※ pages_w1.json をこのノートブックと同じフォルダに置いてから実行してください
with open("pages_w1.json", "r", encoding="utf-8") as f:
    #        ↑ファイル名   ↑「r」=読む(read)  ↑日本語を正しく読むための設定
    loaded_pages = json.load(f)   # ← ファイルの中身をPythonのリストとして読み込む

print(f"読み込んだ件数: {len(loaded_pages)} 件")
print()
print("── 全件のタイトル一覧 ──")
for i, p in enumerate(loaded_pages):
    print(f"  {i+1}件目: {p['title']}  （{p['author']}）")

読み込んだ件数: 5 件

── 全件のタイトル一覧 ──
  1件目: 営業店DX推進レポート  （田中太郎）
  2件目: 住宅ローン審査AI導入の検討  （鈴木花子）
  3件目: 法人融資 稟議書の書き方ガイド  （山本一郎）
  4件目: AIチャットボットによる照会対応DX  （佐藤健太）
  5件目: 全店DX推進ロードマップ2026  （中村美咲）


### コーディング豆知識： `with open(...) as f:` の「with」は何のため？

ファイルを開く操作は **「金庫を開ける」** ようなものです。

- ❌ **悪い例**：金庫を開けたまま作業して **閉め忘れる** → データが壊れる危険がある
- ✅ **良い例**：`with` 構文を使うと、作業が終わったら **自動的に閉じてくれる**

```python
with open("pages_w1.json", "r", encoding="utf-8") as f:
    #                   ↑               ↑
    #              "r" = 読む         日本語を正しく読む
    data = json.load(f)   # ← ここでファイルを読む
# ← ここに来ると自動的にファイルが閉じる
```


## ■ 写経（4）— JSONファイルに書き込む

**このコードでやること：**
- `test_data`（書き込みテスト用のリスト）を用意する
- `json.dump()` を使って `test_output.json` というファイルに書き込む
- 書き込んだファイルを再読み込みして、中身が正しく保存されているか確認する

```
実行すると、こんな出力が得られます：
書き込み完了！ test_output.json の中身：
[
  {
    "id": 99,
    "title": "テスト書き込みページ",
    ...
  }
]
```

**使う知識：** `json.dump()` / `"w"`（書き込みモード） / `ensure_ascii=False`（日本語を文字化けさせない） / `indent=2`（見やすく整形）

> ⚠️ このコードは `test_output.json` という**テスト用ファイル**に書き込みます。  
> `pages_w1.json` は変更されないので、安心して実行してください。


In [4]:
# ── JSONファイルに「書き込む」──
# これが app.py の save_pages() 関数がやっていることです
# 「登録タブ」でフォームを送信したとき、この処理が走ります

test_data = [
    {
        "id": 99,
        "title": "テスト書き込みページ",
        "description": "このデータは書き込みテスト用です",
        "keywords": ["テスト", "Python"],
        "author": "研修受講者",
    }
]

with open("test_output.json", "w", encoding="utf-8") as f:
    #                          ↑
    #              "w" = 書く（write）
    json.dump(test_data, f, ensure_ascii=False, indent=2)
    #                        ↑                   ↑
    #               日本語をそのまま保存       見やすく整形して保存

# 書き込んだ内容を確認
with open("test_output.json", "r", encoding="utf-8") as f:
    content = f.read()

print("書き込み完了！ test_output.json の中身：")
print(content)

書き込み完了！ test_output.json の中身：
[
  {
    "id": 99,
    "title": "テスト書き込みページ",
    "description": "このデータは書き込みテスト用です",
    "keywords": [
      "テスト",
      "Python"
    ],
    "author": "研修受講者"
  }
]


---

## ✅ Step 3 まとめ

**このStepでやったこと**

| やったこと | ひとことで言うと |
|------------|------------------|
| 辞書（dict）を書いた | 1件分のページ ＝ 「キー: 値」のペア |
| リスト（list）にまとめた | 複数件 ＝ `pages_w1.json` の全体像 |
| `json.load()` を使った | ファイル → Python（読み込み）＝ `load_pages()` の中身 |
| `json.dump()` を使った | Python → ファイル（書き込み）＝ `save_pages()` の中身 |
| `with open(...)` を知った | 閉め忘れない安全なファイルの開け方 |

**いまの成果物の状態**

| ファイル | 状態 |
|----------|------|
| `app.py` | 🟡 見た目だけ（変化なし） |
| `pages_w1.json` | ✅ **構造と読み書きを理解**（実ファイルは配布物を使用） |
| `search.py` | ⬜ 未着手（次の Step 4 で着手） |

**➡️ 次は Step 4**：動かなかった原因の2つ目、**探す処理**（`search.py`）を作ります。これで部品が揃います。


---
# Step 4：【ファイル別③】`search.py` — 検索ロジックを作る

> ### 🎯 〈このStepのねらい〉
>
> 見た目（Step 2）とデータ（Step 3）が揃いました。残るは Step 1 の **「🔧 照らし合わせる処理」**。
> 画面の「検索結果：2件」という表示を生み出している、検索エンジンの心臓部を作ります。
>
> **このStepが終わったときにできていること**
> - `def` の **引数と戻り値** を、説明できる
> - `search_pages()` の `for` と `in` を **1行ずつ追える**
> - `highlight_match()` が説明文の語を太字に変える仕組みが言える
>
> **触るファイル**：`search.py`
> （Jupyter で2つの関数を理解 → Step 4 の最後に `search.py` へ書き出します）

## 4-1. 関数（def）とは何か？

**関数とは「繰り返し使える処理のかたまり」に名前をつけたものです**

**自動販売機に例えると——**
- 🪙 **コイン（引数）** を入れて
- 🔘 **ボタン（関数名）** を押すと
- 🥤 **ジュース（戻り値）** が出てくる

**search.py には2つの関数があります**

| 関数名 | 入れるもの（引数） | 出てくるもの（戻り値） | 画面での役割 |
|--------|--------------------|----------------------|--------------|
| `search_pages()` | キーワード＋ページリスト | マッチしたページのリスト | 「検索結果：2件」を作る |
| `highlight_match()` | テキスト＋キーワード | **太字**になったテキスト | 説明文の DX → **DX** |


## ■ 写経（5）— 関数の基本を理解する

**まず `search_pages()` より簡単な関数で「関数とは何か」の構造を体で覚えます**

**このコードでやること：**
- `def 関数名(引数):` の書き方で、シンプルな `say_hello()` 関数を定義する
- 定義した関数を呼び出して、引数によって出力が変わることを確認する
- `return` で値を返す関数と、`print()` するだけの関数の違いを確認する

```
実行すると、こんな出力が得られます：
こんにちは、田中太郎さん！
こんにちは、鈴木花子さん！
```

**使う知識：** `def` による関数定義 / 引数 / `return` / 関数呼び出し


In [5]:
# 【関数の基本】まずシンプルな例で構造を理解しよう

def say_hello(name):                      # def 関数名(引数): で定義する
    message = f"こんにちは、{name}さん！"    # 処理内容（インデント必須）
    return message                         # return で戻り値を返す

# 関数を「呼び出す」
result1 = say_hello("田中太郎")
result2 = say_hello("鈴木花子")

print(result1)
print(result2)
# → 同じ関数を何度でも「別の引数」で使い回せる。これが関数のメリット！

こんにちは、田中太郎さん！
こんにちは、鈴木花子さん！


## 4-2. `search_pages()` を読み解く

**この関数が「検索結果：2件」という表示を生み出しています**

```
┌─────────────────────────────────────────────┐
│  キーワードを入力                              │
│  ┌───────────────────────────────────────┐  │
│  │ DX                                    │  │ ← query = "DX"
│  └───────────────────────────────────────┘  │
│                                             │
│  検索結果：X件    ← len(search_pages(...))    │
└─────────────────────────────────────────────┘
```

## ■ 写経（6）— search_pages() の全体

**このコードでやること：**
- `search_pages(query, pages)` 関数を丸ごと写して定義する
- `"DX"` で検索してヒット件数と各タイトルを表示する
- `"dx"`（小文字）でも同じ結果になることを `.lower()` の効果として確認する

```
実行すると、こんな出力が得られます：
'DX' の検索結果： 4件
  - 営業店DX推進レポート
  - 住宅ローン審査AI導入の検討
  - AIチャットボットによる照会対応DX
  - 全店DX推進ロードマップ2026
（'dx' と小文字で検索しても同じ結果になります）
```

**使う知識：** for ループ / if 文 / `.lower()` / `list.append()` / `return`


In [2]:
# search_pages() の全体 — 1行ずつコメントで解説

def search_pages(query: str, pages: list) -> list:
    # ↑ 「: str」「: list」は型ヒント（なくても動くが読みやすくなる）
    # ↑ 「-> list」は「戻り値はリスト型」という宣言

    # ① キーワードが空欄ならすぐ終了（空リストを返す）
    if not query.strip():
        #  ↑ .strip() は前後の空白を取り除く
        #  ↑ not は「〜でない」 → 空欄なら True になる
        return []

    results = []                    # ② 結果を入れる空のリストを用意
    query_lower = query.lower()    # ③ "DX" → "dx" と小文字に統一（大文字小文字を無視するため）

    for page in pages:              # ④ ページを1件ずつ取り出してループ

        # ⑤ title + description + keywords を1つの文字列に結合して検索対象にする
        search_text = " ".join([
            page["title"],
            page["description"],
            " ".join(page["keywords"]),  # ["DX","営業店"] → "DX 営業店"
        ])
        # 例: "営業店DX推進レポート 窓口業務のペーパーレス化で… DX 営業店 業務効率"

        # ⑥ キーワードが search_text に含まれていたら results に追加
        if query_lower in search_text.lower():
            results.append(page)    # .append() でリストに追加

    return results  # ⑦ マッチしたページのリストを返す


# ── 実際に動かしてみよう ──
import json
with open("pages_w1.json", "r", encoding="utf-8") as f:
    all_pages = json.load(f)

hits = search_pages("DX", all_pages)
print(f"'DX' の検索結果: {len(hits)} 件")
print()
for h in hits:
    print(f"  ✅ {h['title']}  （{h['author']}）")

'DX' の検索結果: 4 件

  ✅ 営業店DX推進レポート  （田中太郎）
  ✅ 住宅ローン審査AI導入の検討  （鈴木花子）
  ✅ AIチャットボットによる照会対応DX  （佐藤健太）
  ✅ 全店DX推進ロードマップ2026  （中村美咲）


### コーディング豆知識「for ループの動きを図解すると？」

```
pages = [田中さん, 鈴木さん, 山本さん, 佐藤さん, 中村さん]  （5件登録済み）

for page in pages:
    ↓ 1周目: page = 田中さんのページ →「DX」含む？ → ✅ → results に追加
    ↓ 2周目: page = 鈴木さんのページ →「DX」含む？ → ✅ → results に追加
    ↓ 3周目: page = 山本さんのページ →「DX」含む？ → ❌ → スキップ
    ↓ 4周目: page = 佐藤さんのページ →「DX」含む？ → ✅ → results に追加
    ↓ 5周目: page = 中村さんのページ →「DX」含む？ → ✅ → results に追加

return [田中, 鈴木, 佐藤, 中村 のページ]  ← 検索結果：4件
```


### `.lower()` がなぜ必要か？

`search_pages()` の中に `query_lower = query.lower()` という行があります。  
これは **「大文字・小文字を区別しないで検索する」** ための処理です。

たとえば利用者が `dx` と小文字で入力しても `DX` を含むページをヒットさせたい。  
`.lower()` がないと、これができません。次のセルを実行して実際に確認してみましょう。


In [3]:
# 【検証】.lower() があるとないとでは何が変わるか体験する

description = "窓口業務のDX推進で事務時間を削減"

# lower() なし：大文字・小文字が違うと一致しない
print("── lower() なし ──")
print("'DX' in text :", "DX" in description)   # True
print("'dx' in text :", "dx" in description)   # False ← 見つからない！
print()

# lower() あり：すべて小文字に揃えて比較するので確実に見つかる
print("── lower() あり ──")
print("'DX' in text :", "DX".lower() in description.lower())  # True
print("'dx' in text :", "dx".lower() in description.lower())  # True ← 見つかる！

── lower() なし ──
'DX' in text : True
'dx' in text : False

── lower() あり ──
'DX' in text : True
'dx' in text : True


## 4-3. `highlight_match()` を読み解く

**この関数が「説明文の中の DX を 太字 に変える」処理をしています**

```
変換前: 窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録

変換後: 窓口業務のペーパーレス化で事務時間を30%削減した**DX**プロジェクトの記録
                          ↑
            Streamlit画面では「太字」で表示される
```

**`re`（正規表現）というライブラリを使って「特定パターンの文字列を検索・置換」します**

## ■ 写経（7）— highlight_match() の全体

**このコードでやること：**
- `highlight_match(text, query)` 関数を丸ごと写して定義する
- 説明文に対してキーワードを渡すと `**キーワード**` に変換されることを確認する
- 大文字 `"DX"` でも小文字 `"dx"` でも同じ結果になることを確認する

```
実行すると、こんな出力が得られます：
【変換前】
  窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録
【変換後 DX】
  窓口業務のペーパーレス化で事務時間を30%削減した**DX**プロジェクトの記録
【変換後 dx】
  窓口業務のペーパーレス化で事務時間を30%削減した**dx**プロジェクトの記録
```

**使う知識：** `import re` / `re.compile()` / `re.IGNORECASE` / `pattern.sub()`


In [4]:
import re  # re = Regular Expression（正規表現）標準ライブラリ、pip install 不要

def highlight_match(text: str, query: str) -> str:

    if not query:       # キーワードが空なら何もしない
        return text

    # re.compile() でパターン（検索ルール）を作る
    pattern = re.compile(
        re.escape(query),  # re.escape：「?」「+」などの特殊文字が含まれても壊れない安全策
        re.IGNORECASE      # IGNORECASE：大文字・小文字を区別しない
    )

    # pattern.sub(置換後の文字列, 元テキスト)：マッチした部分を置換する
    return pattern.sub(f"**{query}**", text)
    #                  ↑
    #   「**DX**」← Markdown で ** で囲むと太字になる！


# ── 動かしてみよう ──
original = "窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録"
result1  = highlight_match(original, "DX")
result2  = highlight_match(original, "dx")   # 小文字でもOK（IGNORECASE のおかげ）

print("【変換前】")   ; print(" ", original)
print("【変換後 DX】") ; print(" ", result1)
print("【変換後 dx】") ; print(" ", result2)
print()
print("※ Streamlit画面では **DX** の部分が太字で表示されます")

【変換前】
  窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録
【変換後 DX】
  窓口業務のペーパーレス化で事務時間を30%削減した**DX**プロジェクトの記録
【変換後 dx】
  窓口業務のペーパーレス化で事務時間を30%削減した**dx**プロジェクトの記録

※ Streamlit画面では **DX** の部分が太字で表示されます


---
## ✅ Step 4 まとめ — Jupyter で学んだ関数を `search.py` に書き出そう

Step 4 では Jupyter 上で **2つの関数** を理解しました。

| 関数 | 役割 |
|------|------|
| `search_pages(query, pages)` | キーワードでページを絞り込む |
| `highlight_match(text, query)` | 説明文のキーワードを太字にする |

この2つは **Jupyter の中だけで動く練習コード** です。  
Streamlit アプリ（`app.py`）から使うには、**`search.py` というファイルに書き出す**必要があります。

```
Jupyter（このノートブック）             search.py
─────────────────────────────   →   ──────────────────────────────
search_pages() を理解した         →   search.py に同じ関数を書く
highlight_match() を理解した      →   search.py に同じ関数を書く
                                               ↓
                                     app.py から
                                     from search import search_pages, highlight_match
                                     で呼び出して使う
```

> 💡 Jupyter で「動く」を確認したコードをそのままコピーして `search.py` に貼れば OK です。  
> 新しいことを覚える必要はありません。**場所を移すだけ**です。
>
> ✍️ **配布の `search.py`（虫食い）** を使うと穴埋め形式で進められます。詰まったら **`search_answer.py`（答え）** と見比べましょう。

---

**いまの成果物の状態**

| ファイル | 状態 |
|----------|------|
| `app.py` | 🟡 見た目だけ（本物の検索はまだ未接続） |
| `pages_w1.json` | ✅ 構造と読み書きを理解 |
| `search.py` | ✅ **2つの関数が完成**（`app.py` からはまだ呼ばれていない） |

**➡️ 次は Step 5**：3つの部品を1本につないで、`app.py` のダミーを本物に差し替えます。


---
# Step 5：3つのファイルを統合して `app.py` を完成させる

> ### 🎯 〈このStepのねらい〉
>
> 部品が3つとも揃いました。Step 2 で置いた **ダミーを本物に差し替えて**、
> バラバラだった3ファイルを **1本の動くアプリ** につなぎます。
>
> **このStepが終わったときにできていること**
> - `app.py` から `search.py` と `pages_w1.json` を呼び出す形が書ける
> - 登録した内容が一覧に反映されるのに `st.cache_data.clear()` が要る理由が言える
> - `st.rerun()` で画面が作り直されること、その前の表示が消えることを説明できる
> - `st.session_state` が「走り直しても残る唯一の入れ物」だと言える
> - `tech0-search/` フォルダが **提出できる完成形** になっている
>
> **触るファイル**：`app.py`（完成させる）＋ `search.py` / `requirements.txt` / `pages_w1.json` の配置

### ダミーを「本物」に差し替える

ここまでで材料は揃いました。

- ✅ **見た目**（Step 2 で作った `app.py` の画面）
- ✅ **データの器**（Step 3 で作った `pages_w1.json`）
- ✅ **検索の処理**（Step 4 で作った `search.py` の2つの関数）

あとは、Step 2 で置いた **ダミーの検索結果カード** を、`search_pages()` が返す **本物の結果** に差し替えるだけです。  
下の骨格を見ながら、`app.py` を完成させましょう。


## app.py の骨格 — ダミーを「本物」に差し替える

**Step 2 で置いたダミーの検索結果を、いま作った `search_pages()` の本物の結果に差し替えます**
**Step 3・4 で学んだJSON操作・検索ロジックが app.py の中でこう使われます**
**「どの順番で何を書くか」の骨格を示します**

```python
# ① ライブラリのインポート
import streamlit as st
import json
from search import search_pages, highlight_match   # Step4 で学んだ関数！

# ② データの読み書き関数（Step3 で学んだ json.load / json.dump を使う）
@st.cache_data
def load_pages(): ...
def save_pages(pages): ...

# ③ ページ設定・タイトル
st.set_page_config(page_title='Tech0 Search v0.1')
st.title('...')

# ④ タブを作る
tab1, tab2, tab3 = st.tabs(['検索', '登録', '一覧'])
pages = load_pages()

# ⑤ 検索タブ
with tab1:
    query = st.text_input('キーワードを入力')
    if query:
        results = search_pages(query, pages)           # Step4 の関数をそのまま使う！
        for page in results:
            st.markdown(highlight_match(               # Step4 の関数をそのまま使う！
                page['description'], query))

# ⑥ 登録タブ
with tab2:
    with st.form('register_form'):
        ...    # 入力フィールドを作る
        submitted = st.form_submit_button('登録する')
    if submitted:
        pages.append(new_page)
        save_pages(pages)                       # Step3 で学んだ json.dump を使う！
        st.cache_data.clear()
        st.session_state['registered'] = True   # 「登録した」印を残す
        st.rerun()                              # 画面を作り直すに反映

# ⑦ 一覧タブ
with tab3:
    for page in pages:
        with st.expander(page['title']):
            ...    # ページ情報を表示
```

> 💡 **Step 3・4 で書いたコードがそのまま活躍します！**  
> `search.py` に書いた2つの関数は `from search import ...` の1行で読み込めます。


## 5-1. 登録処理のポイント — `st.rerun()` と `st.session_state`

### まず大前提：Streamlit は「毎回、上から下まで走り直す」

普通のプログラムは1回動いて終わりですが、**Streamlit は違います。**
ボタンを押す・文字を入力する——何か操作するたびに、`app.py` が **先頭から丸ごと実行し直されます**。

```
ユーザーが操作
     ↓
app.py を1行目から最後まで実行
     ↓
その結果を画面に描く
```

この性質を知らないと、次のような「動いているのに、なぜか表示されない」に必ずぶつかります。

### 登録ボタンを押したときの流れ

```
① new_page = { ... }                   新しいページ辞書を作る
       ↓
② pages.append(new_page)               リストに追加する
       ↓
③ save_pages(pages)                    pages_w1.json に保存（Step3 の json.dump）
       ↓
④ st.cache_data.clear()                古いキャッシュを捨てる
       ↓
⑤ st.session_state['registered'] = True   「登録した」という印を残す
       ↓
⑥ st.rerun()                           ここで実行を打ち切り、①の前まで巻き戻る
       ↓
   ── app.py が最初から走り直す ──
       ↓
⑦ 印を見つけて st.success('登録完了！')  ここで初めて画面に出る
```

### ⚠️ よくある失敗：`st.success()` を `st.rerun()` の前に書く

```python
st.success("登録完了！")   # ← 描画される
st.rerun()                # ← 直後に全部作り直すので、消える
```

**`st.rerun()` は「今の実行を打ち切って、最初からやり直す」命令です。**
それより前に表示したものは **すべて破棄されます。**
JSONへの保存は終わっているのに、**「登録完了！」だけが一瞬も見えない** ——という状態になります。

### だから `st.session_state` を使う

走り直すたびに、**普通の変数は全部リセットされます。**
`registered = True` と書いても、次の実行では消えています。

**唯一、走り直しても消えないのが `st.session_state` です。**

| | 走り直すとどうなる？ | たとえると |
|---|---|---|
| 普通の変数（`x = 1`） | **消える**（毎回作り直される） | 机の上の書きかけのメモ |
| `st.session_state['x'] = 1` | **残る** | 机に貼った付箋 |

```python
# ── 登録できたときに「印」を残す ──
st.session_state['registered'] = True
st.rerun()

# ── 走り直した先で、印を見つけたらメッセージを出す ──
if st.session_state.get("registered"):
    st.success("登録完了！")
    st.session_state["registered"] = False   # 一度出したら印を消す
```

> 💡 **`st.session_state` は「ブラウザのタブごとに用意されたメモ帳」** です。
> タブを開いている間はずっと残り、閉じると消えます。
> 別の人が別のタブで開けば、その人には別のメモ帳が用意されます。
>
> 📌 `.get("registered")` を使っているのは、**まだ一度も登録していないとき**（キーが存在しないとき）に
> エラーにせず「無い＝False」として扱うためです。辞書の `.get()` と同じ考え方です（Step 3 で学びました）。

### `st.cache_data.clear()` は何のため？

`load_pages()` には `@st.cache_data` が付いていて、**一度読んだJSONを覚えています**。
そのままだと、走り直しても **古い内容を返してしまう** ので、
保存した直後に「覚えていた内容を忘れて」と指示するのが `st.cache_data.clear()` です。

**この3つはセットで覚えてください。**

| 命令 | 役割 |
|------|------|
| `st.cache_data.clear()` | 覚えていた古いデータを捨てる |
| `st.session_state[...]` | 走り直しても消したくない情報を残す |
| `st.rerun()` | 画面を最初から作り直す |


---
## 5-2. いよいよ `app.py` を書こう！

> 📁 **フォルダ構成は Step 1（1-3）、`app.py` の中身は Step 5 の骨格** を見ればわかります（重複するのでここでは繰り返しません）。

---

### 〈Jupyter とアプリフォルダの役割の違い〉

| このノートブック（Jupyter） | tech0-search/（アプリフォルダ） |
|---------------------------|--------------------------------|
| コードを **理解する** 場所 | コードを **動かす** 場所 |
| 1セルずつ実行して確認する | `streamlit run app.py` で一括起動 |
| 練習・実験用 | 成果物・提出用 |

---

### 〈作業手順〉

```
① tech0-search/ フォルダを新しく作る
       ↓
② search.py を作る
   Step 4 の search_pages() と highlight_match() をそのままコピーして貼る
       ↓
③ requirements.txt を作る
   中身に「streamlit>=1.32.0」と書いて保存する
       ↓
④ pages_w1.json を配布フォルダ（このノートブックと同じ場所）からコピーする
       ↓
⑤ app.py を書く  ← ここが今週のメイン！
   Step 5「app.py の骨格」を参考にしながら自分で作る
       ↓
⑥ ターミナルでライブラリをインストールする
   pip install -r requirements.txt
       ↓
⑦ Step 6 の練習問題で理解を確認する
       ↓
⑧ Step 7 で streamlit run app.py を実行して動かす！
```

> 💡 **迷ったときは**  
> 「Jupyter で動いた」→ そのコードを `.py` ファイルに書き出す、これだけです。  
> Step 5 の骨格とコマンド早見表を手元に置きながら進めましょう。  
> どうしても動かないときは **`answers/w1/`（完成コード）** と見比べてください。


---

## ✅ Step 5 まとめ

**このStepでやったこと**

| やったこと | ひとことで言うと |
|------------|------------------|
| app.py の骨格を読んだ | ①インポート 〜 ⑦一覧タブ までの全体像 |
| ダミーを本物に差し替えた | `search_pages()` の戻り値を画面に流す |
| Streamlitが毎回走り直すことを知った | 操作のたびに `app.py` が先頭から再実行される |
| `st.session_state` を使った | 走り直しても消えない「付箋」。`st.rerun()` をまたいでメッセージを渡す |
| フォルダを完成形にした | `tech0-search/` に4ファイルを配置 |

**いまの成果物の状態**

| ファイル | 状態 |
|----------|------|
| `app.py` | ✅ **完成**（起動確認は Step 7） |
| `pages_w1.json` | ✅ 配布物を配置済み |
| `search.py` | ✅ 2つの関数を書き出し済み |

**➡️ 次は Step 6**：練習問題4問で、ここまでの理解が本物か確認します。


---
# Step 6：練習問題で総まとめ

> ### 🎯 〈このStepのねらい〉
>
> 「読んで分かった」と「自分で書ける」は別物です。
> Step 3・4・5 で学んだことが本当に身についているか、手を動かして確認します。
>
> **このStepが終わったときにできていること**
> - JSONを読み込み、**条件で絞り込んで** 表示できる（練習1）
> - 自分で定義した関数を **呼び出して** 結果を加工できる（練習2・4）
> - `pages_w1.json` に **自分のデータを追記** できる（練習3）
>
> **触るファイル**：このノートブックのコードセル（練習3のみ `pages_w1.json` を更新します）

## ■ 練習問題

ここまでで学んだJSON操作・関数・検索ロジック・Streamlit構造を使って問題を解きましょう！


---
**練習1**

`pages_w1.json` を読み込んで、**カテゴリが「企画」のページだけ** タイトルと作者を表示してください。

```
期待する出力例：
カテゴリ: 企画 のページ一覧
────────────────────────────────────────
  1件目: （タイトル）  （作者）
  2件目: ...
合計: ○ 件
```

**使う知識：** `json.load()` / for ループ / `page["category"]` の条件分岐

> 💡 **ヒント**：練習3を複数回実行していると `KeyError` が出ることがあります。  
> その場合は配布された元の `pages_w1.json` に差し替えてから再実行してください。


In [6]:
# ── 練習1 解答欄（直下に模範解答があります！） ──
import json
with open("pages_w1.json", "r", encoding="utf-8") as f:
    pages = json.load(f)

print("カテゴリ: 企画 のページ一覧")
print("────────────────────────────────────────")
count = 0
for page in pages:
    if page.get('category') == '企画':
        count += 1
        print(f" {count}件目: {page['title']}    {page['author']}")
print(f"\n合計: {count} 件")























カテゴリ: 企画 のページ一覧
────────────────────────────────────────
 1件目: 住宅ローン審査AI導入の検討    鈴木花子
 2件目: 全店DX推進ロードマップ2026    中村美咲

合計: 2 件


In [7]:
# ── 練習1 模範解答 ──
import json

with open('pages_w1.json', 'r', encoding='utf-8') as f:
    pages = json.load(f)

print('カテゴリ: 企画 のページ一覧')
print('-' * 40)
count = 0
for page in pages:
    if page.get('category', '') == '企画':
        count += 1
        print(f"  {count}件目: {page['title']}  （{page['author']}）")
print(f'\n合計: {count} 件')


カテゴリ: 企画 のページ一覧
----------------------------------------
  1件目: 住宅ローン審査AI導入の検討  （鈴木花子）
  2件目: 全店DX推進ロードマップ2026  （中村美咲）

合計: 2 件


---
**練習2**

`search_pages()` を使って **「融資」で検索し**、ヒットした各ページの  
タイトル・作者・キーワードを表示してください。

```
期待する出力例：
'融資' の検索結果: ○ 件
────────────────────────────────────────
  タイトル  : （タイトル）
  作者      : （作者）
  キーワード: AI, 融資, 審査
```

**使う知識：** Step 4 で定義した `search_pages()` / for ループ


In [10]:
# ── 練習2 解答欄（直下に模範解答があります！） ──

# search_pages() の全体 — 1行ずつコメントで解説

def search_pages(query: str, pages: list) -> list:
    # ↑ 「: str」「: list」は型ヒント（なくても動くが読みやすくなる）
    # ↑ 「-> list」は「戻り値はリスト型」という宣言

    # ① キーワードが空欄ならすぐ終了（空リストを返す）
    if not query.strip():
        #  ↑ .strip() は前後の空白を取り除く
        #  ↑ not は「〜でない」 → 空欄なら True になる
        return []

    results = []                    # ② 結果を入れる空のリストを用意
    query_lower = query.lower()    # ③ "DX" → "dx" と小文字に統一（大文字小文字を無視するため）

    for page in pages:              # ④ ページを1件ずつ取り出してループ

        # ⑤ title + description + keywords を1つの文字列に結合して検索対象にする
        search_text = " ".join([
            page["title"],
            page["description"],
            " ".join(page["keywords"]),  # ["DX","営業店"] → "DX 営業店"
        ])
        # 例: "営業店DX推進レポート 窓口業務のペーパーレス化で… DX 営業店 業務効率"

        # ⑥ キーワードが search_text に含まれていたら results に追加
        if query_lower in search_text.lower():
            results.append(page)    # .append() でリストに追加

    return results  # ⑦ マッチしたページのリストを返す


# ── 実際に動かしてみよう ──
import json
with open("pages_w1.json", "r", encoding="utf-8") as f:
    all_pages = json.load(f)

hits = search_pages("融資", all_pages)
print(f"'融資' の検索結果: {len(hits)} 件")
print()
for h in hits:
    print(f"  ✅ {h['title']}  （{h['author']}）　({', '.join(h['keywords'])})")


























'融資' の検索結果: 2 件

  ✅ 住宅ローン審査AI導入の検討  （鈴木花子）　(AI, 融資, 審査)
  ✅ 法人融資 稟議書の書き方ガイド  （山本一郎）　(融資, 稟議, 法人)


In [9]:
# ── 練習2 模範解答 ──
import json

with open('pages_w1.json', 'r', encoding='utf-8') as f:
    pages = json.load(f)

def search_pages(query, pages):
    if not query.strip():
        return []
    results = []
    query_lower = query.lower()
    for page in pages:
        search_text = ' '.join([
            page['title'],
            page['description'],
            ' '.join(page['keywords']),
        ])
        if query_lower in search_text.lower():
            results.append(page)
    return results

hits = search_pages('融資', pages)
print(f"'融資' の検索結果: {len(hits)} 件")
print('-' * 40)
for h in hits:
    print(f"  タイトル  : {h['title']}")
    print(f"  作者      : {h['author']}")
    print(f"  キーワード: {', '.join(h['keywords'])}")
    print()


'融資' の検索結果: 2 件
----------------------------------------
  タイトル  : 住宅ローン審査AI導入の検討
  作者      : 鈴木花子
  キーワード: AI, 融資, 審査

  タイトル  : 法人融資 稟議書の書き方ガイド
  作者      : 山本一郎
  キーワード: 融資, 稟議, 法人



---
**練習3**

**自分のLP、または行内で共有したいページ**の情報で新しいページ辞書を作り、`pages_w1.json` に追記してください。  
実行後は `pages_w1.json` を直接開いて、自分のデータが追加されているか確認しましょう。

```
設定する項目：
  url         : あなたのLP または共有したいページの URL
  title       : ページタイトル
  description : 説明文
  keywords    : キーワードのリスト
  author      : あなたの名前
  category    : カテゴリ名
```

**使う知識：** `json.load()` / 辞書の作成 / `list.append()` / `json.dump()`

> ⚠️ **注意**：このセルを実行するたびに `pages_w1.json` に1件追加されます。  
> 複数回実行すると同じデータが重複して登録されます。  
> 練習1を実行したときに `KeyError` が出た場合は、配布された元の `pages_w1.json` に差し替えてから再実行してください。


In [12]:
# ── 練習3 解答欄（直下に模範解答があります！） ──
import json

with open('pages_w1.json', 'r', encoding='utf-8') as f:
    pages = json.load(f)
new_page = {
    "id"         : len(pages) + 1,  # 新しいページのIDは既存件数 + 1
    "title"      : "MyProfile",
    "url"        : "https://magenta-lamington-5de516.netlify.app",
    "description": "自己紹介ページ",
    "keywords"   : ["長岡", "新潟", "味噌ラーメン"],   # ← [ ] はリスト（複数の値）
    "created_at" : "2025-12-01",
    "author"     : "石原真奈",
    "category"   : "ブログ",
}

pages.append(new_page)  # 新しいページをリストに追加
with open("pages_w1.json", "w", encoding="utf-8") as f:
    json.dump(pages, f, ensure_ascii=False, indent=2)  # ファイルに書き込む

print(f"新しいページを追加しました: 現在の件数 = {len(pages)} 件")
print(f"追加したページのタイトル: {new_page['title']}")
print()
print("--追加された1件--")
with open("pages_w1.json", "r", encoding="utf-8") as f:
    updated = json.load(f)
for k, v in updated[-1].items():  # updated[-1] = 最後に追加された1件
    print(f"  {k}: {v}")



























新しいページを追加しました: 現在の件数 = 6 件
追加したページのタイトル: MyProfile

--追加された1件--
  id: 6
  title: MyProfile
  url: https://magenta-lamington-5de516.netlify.app
  description: 自己紹介ページ
  keywords: ['長岡', '新潟', '味噌ラーメン']
  created_at: 2025-12-01
  author: 石原真奈
  category: ブログ


In [ ]:
# ── 練習3 模範解答 ──
import json
from datetime import datetime

with open('pages_w1.json', 'r', encoding='utf-8') as f:
    pages = json.load(f)

new_page = {
    'id'         : len(pages) + 1,
    'url'        : 'https://your-page.example.com',
    'title'      : 'あなたのページのタイトル',
    'description': 'ここに説明文を書いてください',
    'keywords'   : ['キーワード1', 'キーワード2'],
    'author'     : 'あなたの名前',
    'created_at' : str(datetime.today().date()),
    'category'   : '共有',
}

pages.append(new_page)
with open('pages_w1.json', 'w', encoding='utf-8') as f:
    json.dump(pages, f, ensure_ascii=False, indent=2)

print(f'登録完了！ 現在の件数: {len(pages)} 件')
print(f'追加したタイトル: {new_page["title"]}')
print()
print('── 追加された1件 ──')
with open('pages_w1.json', 'r', encoding='utf-8') as f:
    updated = json.load(f)
for k, v in updated[-1].items():
    print(f'  {k}: {v}')


---
**練習4（応用）**

`search_pages()` と `highlight_match()` を **組み合わせて**、  
**「AI」で検索した結果**を表示してください。  
説明文の中の「AI」が `**AI**` に変換されていることを確認しましょう。

```
期待する出力例：
'AI' の検索結果: ○ 件
───────────────────────────────────────────────────────
タイトル    : （タイトル）
説明（変換後）: ...（ **AI** の部分が太字マークに変わっている）...
キーワード  : `AI`, `DX`
作者        : （作者）
```

**使う知識：** `search_pages()` / `highlight_match()` / for ループ


In [13]:
# ── 練習4 解答欄（直下に模範解答があります！） ──
import json, re

with open('pages_w1.json', 'r', encoding='utf-8') as f:
    pages = json.load(f)

def search_pages(query: str, pages: list) -> list:
    if not query.strip():
        return []
    results = []
    query_lower = query.lower()
    for page in pages:
        search_text = " ".join([
            page["title"],
            page["description"],
            " ".join(page["keywords"]),
        ])
        if query_lower in search_text.lower():
            results.append(page)
    return results

def highlight_match(text: str, query: str) -> str:
    if not query:
        return text
    pattern = re.compile(re.escape(query), re.IGNORECASE)
    return pattern.sub(f"**{query}**", text)

query = "AI"
hits = search_pages(query, pages)
print(f"検索キーワード: '{query}'  →  {len(hits)} 件")
print()
for i, h in enumerate(hits):
    print(f"  {i+1}件目: {highlight_match(h['title'], query)}  （{h['author']}）")
    print(f"    説明: {highlight_match(h['description'], query)}")
    print(f"    キーワード: {', '.join([highlight_match(k, query) for k in h['keywords']])}")
    print()



























検索キーワード: 'AI'  →  2 件

  1件目: 住宅ローン審査**AI**導入の検討  （鈴木花子）
    説明: **AI**事前審査で融資業務をDX化する企画提案
    キーワード: **AI**, 融資, 審査

  2件目: **AI**チャットボットによる照会対応DX  （佐藤健太）
    説明: コールセンターに**AI**チャットボットを導入し照会対応を効率化したDX事例
    キーワード: **AI**, DX, コールセンター



In [ ]:
# ── 練習4 模範解答 ──
import json, re

with open('pages_w1.json', 'r', encoding='utf-8') as f:
    pages = json.load(f)

def search_pages(query, pages):
    if not query.strip(): return []
    results = []
    query_lower = query.lower()
    for page in pages:
        search_text = ' '.join([page['title'], page['description'],
                                 ' '.join(page['keywords'])])
        if query_lower in search_text.lower():
            results.append(page)
    return results

def highlight_match(text, query):
    if not query: return text
    pattern = re.compile(re.escape(query), re.IGNORECASE)
    return pattern.sub(f'**{query}**', text)

query = 'AI'
hits  = search_pages(query, pages)
print(f"'{query}' の検索結果: {len(hits)} 件")
print('-' * 55)
for page in hits:
    print(f"タイトル    : {page['title']}")
    print(f"説明（変換後）: {highlight_match(page['description'], query)}")
    tags = ', '.join([f'`{kw}`' for kw in page['keywords']])
    print(f"キーワード  : {tags}")
    print(f"作者        : {page['author']}")
    print()


---

## ✅ Step 6 まとめ

**このStepでやったこと**

| やったこと | ひとことで言うと |
|------------|------------------|
| 練習1 | JSONの読み込み ＋ 条件で絞り込み（Step 3 の確認） |
| 練習2 | 自作関数 `search_pages()` の呼び出し（Step 4 の確認） |
| 練習3 | `pages_w1.json` への追記＝登録機能の中身（Step 3・5 の確認） |
| 練習4 | 2つの関数の組み合わせ（Step 4 の総合確認） |

**いまの成果物の状態**

| ファイル | 状態 |
|----------|------|
| `app.py` | ✅ 完成 |
| `pages_w1.json` | ✅ 自分のデータも追記済み |
| `search.py` | ✅ 完成 |

**➡️ 次は Step 7**：いよいよ `streamlit run app.py`。アプリを起動して CDO レビューへ。


---
# Step 7：アプリを起動して CDO レビューへ！

> ### 🎯 〈このStepのねらい〉
>
> コードは完成しました。最後は **実際に起動して、CDOに見せられる状態か検証** します。
> 動かして初めて分かる不具合と、v0.1 の「限界」を、ここで自分の目で確かめます。
>
> **このStepが終わったときにできていること**
> - `streamlit run app.py` でアプリを起動できる
> - 動作確認チェックリスト7項目を **すべて ✅ にできる**
> - 20件登録してみて、**この検索の弱点** に自分で気づける
>
> **触るファイル**：ターミナルとブラウザ（コード修正は不具合が出たときだけ）

ここまでで **Tech0 Search v0.1 のコード全体** を読み解きました。  
いよいよ実際にアプリを起動して、黒崎 CDO のレビューに備えましょう！

### 🚀 起動手順

**① ターミナルを開く**（VS Code では `Ctrl + @`）

**② app.py があるフォルダに移動する**
```bash
cd （app.py があるフォルダのパス）
```

**③ アプリを起動する**
```bash
streamlit run app.py
```

**④ ブラウザで `http://localhost:8501` が自動的に開く**

### ✅ 動作確認チェックリスト

| チェック | 確認内容 | 対応するコード |
|----------|----------|---------------|
| ☐ | タイトルが表示されているか | `st.title()` |
| ☐ | タブが3つ表示されているか | `st.tabs()` |
| ☐ | 「DX」で4件ヒットするか | `search_pages()` |
| ☐ | 説明文の「DX」が太字になっているか | `highlight_match()` |
| ☐ | 登録タブで自分のページを登録できるか | `save_pages()` + `st.form()` |
| ☐ | 登録後に **「登録完了！」が表示される**か | `st.session_state` + `st.rerun()` |
| ☐ | 登録後、一覧タブに反映されているか | `st.cache_data.clear()` + `load_pages()` |
| ☐ | pages_w1.json ファイルが更新されているか | `json.dump()` |

### 🚀 動作確認ができたら、ページを20件登録して検索を試してみよう！

① 受講者同士のLPや、その他公開ページ（自社のニュースリリースなど）を合わせて **20件** 登録！

② 検索で想定したページが出てくるかを検証！！

### ✅ Step 7 まとめ — 成果物の最終状態

| ファイル | 状態 |
|----------|------|
| `app.py` | ✅ 起動確認済み（検索・登録・一覧の3タブが動く） |
| `pages_w1.json` | ✅ 20件登録済み |
| `search.py` | ✅ `app.py` から呼ばれて動作中 |
| `requirements.txt` | ✅ 作成済み |

**➡️ この後**：W1のふりかえりで、この v0.1 の「限界」を確認し、W2 の課題につなげます。

### 📬 W1 終了 — CDO レビュー

```
╔════════════════════════════════════════════╗
║  CDO黒崎からのフィードバック                   ║
╠════════════════════════════════════════════╣
║                                            ║
║  デモ、見た。                                ║
║                                            ║
║  キーワード入れたら結果が出る。                 ║
║  いいじゃないか。                             ║
║                                            ║
║  ......ちょっと待て。                        ║
║  このデータ、誰が登録したんだ？                 ║
║                                            ║
║  手動？20件を？                              ║
║                                            ║
║  我が社には3,000人の社員がいる。               ║
║  過去10年分の提案書は何万件だと思う？            ║
║  それを全部、手動で登録するのか？               ║
║                                            ║
║  いや、よくやった。最初の一歩としては            ║
║  十分だ。だが次のステップが見えたな。            ║
║                                            ║
║  「自動化」だ。                              ║
║                                            ║
║  → W2 へ続く...                             ║
╚════════════════════════════════════════════╝
```

**次は W2 で「クローラー（自動収集）」に挑戦します！**（その後 W3 で DB化＋全文検索、W4 でランキング）  
URLを入れたら自動でページ情報を取ってくる仕組みを作ります。

---
# 🧩 W1のふりかえりと、次週への課題

お疲れさまでした。W1で **Tech0 Search v0.1** ができました。ここで一度立ち止まって、いまの検索の「限界」を確認します。

### いま作った検索は「部分一致（簡易検索）」

W1の `search_pages()` は、`title`・`description`・`keywords` に検索語が **含まれているか** を調べているだけです。
これはこれで動きますが、次の弱点があります。

- ❌ **本文（ページの中身）は検索できない**。タイトルに語が無いと、本文にどれだけ出てきてもヒットしない。
- ❌ 登録は **手動**。人がいちいち `pages_w1.json` に打ち込んでいる。数件なら回るが、実務では破綻する。

### 「本文まで検索したい」──でも、その本文はどこから？

本文まで検索できる **全文検索** をやりたい。でも、そもそも各ページの**本文データ（full_text）を持っていない**。
手で全ページの本文をコピペするのは非現実的です。

> 💬 **CDO黒崎の指令（W2へ）**
> 「タイトルだけの検索など検索と呼べん。本文まで探せ。
> ……だが、その本文を **どうやって集める** つもりだ？ 手で貼るのか？ 自動化しろ。」

➡️ この「**何を・どうやって取ってくるか**」という問いへの答えが、次週 **W2 のクローラー** です。
クローラーが本文（full_text）を集めてくれて初めて、**全文検索**が本当の意味で成立します。

ぜひお楽しみに！！！
